This notebook divides breakpoints across genomics bins for Significantly Recurrent Bins Analysis

Outputs:
- binned_breakpoints.csv -> which bin each breakpoint is in (after removing multiple breakpoints per sample per bin)
- genome_bins.bed -> the start and end coordinates of each bin
- breakpoints_per_bin_distribution.csv -> Number of breakpoints within each bin

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import h5py
OUTPUT_PATH = Path("../data")
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
h5_file = "../../SV_sample_data/data/analysis_public_os_data_merged_consensus_svs_v2.h5"
df = pd.read_hdf(h5_file).copy()
included_samples = pd.read_csv("../../SV_sample_data/data/final_analysis_samples.csv")
# remove excluded samples and multiple tumors
df = df[df['name'].isin(included_samples['tumor_normal_pair'])]
print("Number of structural variant calls:", len(df))
print("Number of participants:", len(df['name'].unique()))

Number of structural variant calls: 53885
Number of participants: 205


In [3]:
def load_and_process_blacklist(blacklist_file):
    blacklist = pd.read_csv(blacklist_file, sep='\t', 
                           names=['chrom', 'start', 'end'],
                           compression='gzip')
    blacklist['chrom'] = blacklist['chrom'].str.replace('chr', '')
    return blacklist

def load_and_process_gaps(gap_file):
    """
    column 5 is the component type (N = gap)
    """
    gaps = pd.read_csv(gap_file, sep='\t', 
                      names=['chrom', 'start', 'end', 'part_num', 'component_type', 
                            'component_id', 'gap_type_or_start', 'linkage_or_end', 'orientation_or_evidence'])
    
    # filter for gap regions
    gaps = gaps[gaps['component_type'] == 'N']
    
    # convert from 1-based to 0-based coordinates by subtracting 1 from start
    # (end remains the same as it's exclusive in 0-based)
    gaps['start'] = gaps['start'] - 1
    
    gaps['chrom'] = gaps['chrom'].str.replace('chr', '')
    
    return gaps[['chrom', 'start', 'end']]

def is_bin_problematic(bin_row, blacklist_df, gaps_df):
    """
    Check if a bin overlaps with any blacklisted region or gap
    """
    
    blacklist_overlap = blacklist_df[
        (blacklist_df['chrom'] == bin_row['chrom']) &
        (blacklist_df['start'] < bin_row['end']) &
        (blacklist_df['end'] > bin_row['start'])
    ]
    
    # check gap overlap
    gap_overlap = gaps_df[
        (gaps_df['chrom'] == bin_row['chrom']) &
        (gaps_df['start'] < bin_row['end']) &
        (gaps_df['end'] > bin_row['start'])
    ]
    
    return len(blacklist_overlap) > 0 or len(gap_overlap) > 0

In [4]:
def create_breakpoint_bins(sv_df, chrom_sizes_dict, blacklist_file, gap_file, bin_size=50000, overlap=0):
    """
    Create genomic bins (with or without overlap), excluding problematic regions
    """
    # read the SV file
    df = sv_df
    
    # load the blacklist and gaps
    print("Loading blacklist and gap regions...")
    blacklist = load_and_process_blacklist(blacklist_file)
    gaps = load_and_process_gaps(gap_file)
    
    # chromosome name mapping for X & Y
    chrom_map = {
        '23': 'X',
        '24': 'Y'
    }
    df['chr1'] = df['chr1'].astype(str).map(lambda x: chrom_map.get(x, x))
    df['chr2'] = df['chr2'].astype(str).map(lambda x: chrom_map.get(x, x))
    
    # extract breakpoints (both ends for each SV)
    breakpoints = pd.DataFrame({
        'chrom': pd.concat([df['chr1'], df['chr2']]),
        'position': pd.concat([df['pos1'], df['pos2']]),
        'sample': pd.concat([df['name'], df['name']])
    }).reset_index(drop=True)
    
    # filter out chromosomes not in the size dictionary
    valid_chroms = set(chrom_sizes_dict.keys())
    breakpoints = breakpoints[breakpoints['chrom'].isin(valid_chroms)]

    bins_list = []
    bin_counter = 0
    
    # currently non-overlapping bins, but can change to overlapping bins if needed
    # for fishhook, we use overlap = 0, since Fishhook assumes bin independence for significance testing
    # can try running fishhook twice with different starting points and taking intersection of significant bins
    print("Creating bins and filtering problematic regions...")
    for chrom, size in chrom_sizes_dict.items():
        start_offset = 25000
        step_size = bin_size - overlap
        starts = np.arange(start_offset, size - bin_size + 1, step_size)
        ends = starts + bin_size
        
        # step_size = bin_size - overlap
        # n_bins = (size - overlap) // step_size + 1
        # starts = np.arange(n_bins) * step_size
        # ends = np.minimum(starts + bin_size, size)
        
        for start, end in zip(starts, ends):
            bin_data = {
                'bin_id': bin_counter,
                'chrom': chrom,
                'start': start,
                'end': end
            }
            
            # only add the bin if it doesn't overlap with problematic regions
            if not is_bin_problematic(bin_data, blacklist, gaps):
                bins_list.append(bin_data)
                bin_counter += 1
    
    bins = pd.DataFrame(bins_list)
    print(f"Created {len(bins)} bins after filtering problematic regions")
    
    # create list to store breakpoint-bin assignments
    assignments = []
    
    for chrom in valid_chroms:
        # brkpts and bins for this chromosome
        chrom_breakpoints = breakpoints[breakpoints['chrom'] == chrom]
        chrom_bins = bins[bins['chrom'] == chrom]
        
        if len(chrom_breakpoints) == 0 or len(chrom_bins) == 0:
            continue
        
        # for each bin, find breakpoints that fall within it
        for _, bin_row in chrom_bins.iterrows():
            mask = (chrom_breakpoints['position'] >= bin_row['start']) & \
                  (chrom_breakpoints['position'] < bin_row['end'])
            
            if mask.any():
                matching_breakpoints = chrom_breakpoints[mask]
                for _, bp_row in matching_breakpoints.iterrows():
                    assignments.append({
                        'bin_id': bin_row['bin_id'],
                        'chrom': bp_row['chrom'],
                        'position': bp_row['position'],
                        'sample': bp_row['sample']
                    })
    
    breakpoints_with_bins = pd.DataFrame(assignments)
    
    # keep only one breakpoint per bin per sample
    unique_breakpoints = breakpoints_with_bins.drop_duplicates(subset=['bin_id', 'sample'])
    
    # stats
    breakpoints_per_bin = unique_breakpoints.groupby('bin_id').size()
    total_bins = len(bins)
    bins_with_breakpoints = len(breakpoints_per_bin)
    
    print("\nBin Statistics:")
    print(f"Total number of bins: {total_bins}")
    print(f"Bins with breakpoints: {bins_with_breakpoints}")
    print(f"Percentage of bins with breakpoints: {(bins_with_breakpoints/total_bins)*100:.2f}%")

    print("\nBreakpoints per bin statistics:")
    print(f"Mean breakpoints per bin: {breakpoints_per_bin.mean():.2f}")
    print(f"Median breakpoints per bin: {breakpoints_per_bin.median():.2f}")
    print(f"Max breakpoints in a bin: {breakpoints_per_bin.max()}")
    print(f"Min breakpoints in a bin (excluding empty bins): {breakpoints_per_bin.min()}")
    
    print(f"\nBreakpoint statistics:")
    print(f"Total number of original breakpoints: {len(breakpoints)}")
    print(f"Number of breakpoints after binning: {len(unique_breakpoints)}")
    print(f"Number of samples: {len(unique_breakpoints['sample'].unique())}")

    print("\nFiltering Statistics:")
    print(f"Number of blacklist regions: {len(blacklist)}")
    print(f"Number of gap regions: {len(gaps)}")

    # outputs
    unique_breakpoints.to_csv(os.path.join(OUTPUT_PATH, 'binned_breakpoints.csv'), index=False)
    bins.to_csv(os.path.join(OUTPUT_PATH, 'genome_bins.bed'), sep='\t', index=False)
    breakpoints_per_bin.to_csv(os.path.join(OUTPUT_PATH, 'breakpoints_per_bin_distribution.csv'))
    
    return bins, unique_breakpoints

# ENCODE
blacklist_file = "../data/hg38_ref_files/ENCFF356LFX.bed.gz"
# UCSC
gap_file = "../data/hg38_ref_files/hg38.agp.gz"
# UCSC chromosome sizes
chrom_sizes_dict = {
    '1': 248956422, '2': 242193529, '3': 198295559, 
    '4': 190214555, '5': 181538259, '6': 170805979, 
    '7': 159345973, '8': 145138636, '9': 138394717, 
    '10': 133797422, '11': 135086622, '12': 133275309, 
    '13': 114364328, '14': 107043718, '15': 101991189, 
    '16': 90338345, '17': 83257441, '18': 80373285, 
    '19': 58617616, '20': 64444167, '21': 46709983, 
    '22': 50818468, 'X': 156040895, 'Y': 57227415
    }

bins, binned_breakpoints = create_breakpoint_bins(
    df, 
    chrom_sizes_dict,
    blacklist_file=blacklist_file,
    gap_file=gap_file
)

Loading blacklist and gap regions...
Creating bins and filtering problematic regions...
Created 56596 bins after filtering problematic regions

Bin Statistics:
Total number of bins: 56596
Bins with breakpoints: 35772
Percentage of bins with breakpoints: 63.21%

Breakpoints per bin statistics:
Mean breakpoints per bin: 1.84
Median breakpoints per bin: 1.00
Max breakpoints in a bin: 91
Min breakpoints in a bin (excluding empty bins): 1

Breakpoint statistics:
Total number of original breakpoints: 107770
Number of breakpoints after binning: 65661
Number of samples: 205

Filtering Statistics:
Number of blacklist regions: 910
Number of gap regions: 819
